In [1]:
import cv2
import torch
import numpy as np
from collections import deque
from ultralytics import YOLO
from catboost import CatBoostRegressor

In [ ]:
dataset_rows = []
model = YOLO(r'C:\Users\user\projectolymp\runs\detect\models\111\weights\best.pt')
cap = cv2.VideoCapture(0)
track_history = {}
cnt = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        break
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", conf = 0.5) # ищет маркеры и присваевает им id
    if results[0].boxes.id is not None: # провеййрка нашла ли модель хоть 1 маркер
        boxes = results[0].boxes.xywh.cpu().numpy() # получение координатов объектов
        track_ids = results[0].boxes.id.int().cpu().tolist() # получение id
        for box, track_id in zip(boxes, track_ids): # проходим по координате и ее id
            x, y, w, h = box # распаковка
            track = track_history.get(track_id, deque(maxlen=150)) # история 5 секунд при 30 фпс
            track.append((float(x), float(y))) # добавляем точку в историю движения 
            track_history[track_id] = track # сохраняем обновленную историю
            if len(track) >= 25: # Изменил на >=, чтобы индекс -25 работал корректно
                past = list(track)[-25:-15] 
                future = track[-1]
                row = [coord for pt in past for coord in pt] + [float(future[0]), float(future[1])]
                dataset_rows.append(row) 
            for i in range(1, len(track)): # отрисовка
                pt1 = (int(track[i-1][0]), int(track[i-1][1]))
                pt2 = (int(track[i][0]), int(track[i][1]))
                cv2.line(frame, pt1, pt2, (0, 0, 255), 2) # красная линия толщиной 2 пикселя
        cnt += 1
    cv2.imshow("MMC", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): # на q закрытие
        break

cap.release()
cv2.destroyAllWindows()

if len(dataset_rows) > 0:
    data_array = np.array(dataset_rows, dtype=np.float32)
    X = data_array[:, :-2]
    Y = data_array[:, -2:]
    cb_model = CatBoostRegressor(iterations=1000, 
                                 loss_function='MultiRMSE', 
                                 learning_rate=0.1,
                                 task_type="CPU") # Можно сменить на GPU, если есть
    cb_model.fit(X, Y)
    cb_model.save_model("catboost.cbm")
else:
    print("Данных для обучения не собрано!")


0: 480x640 3 markers, 14.1ms
Speed: 46.4ms preprocess, 14.1ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 marker, 13.4ms
Speed: 2.9ms preprocess, 13.4ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 markers, 10.4ms
Speed: 2.3ms preprocess, 10.4ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 markers, 13.8ms
Speed: 5.4ms preprocess, 13.8ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 markers, 16.5ms
Speed: 1.8ms preprocess, 16.5ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 markers, 12.2ms
Speed: 2.0ms preprocess, 12.2ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 markers, 12.7ms
Speed: 1.9ms preprocess, 12.7ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 markers, 11.8ms
Speed: 2.3ms preprocess, 11.8ms inference, 1.1ms postprocess per image at shape (